# Notebook 22 (v2) — بنچمارک روی Matbench (matbench_phonons)

## هدف
تست همون انکودر دوگانه‌ی گراف (bond graph + atom graph) پروژه — که برای IFC فازهای MAX
ساخته شده بود — روی تسک عمومی‌تر **`matbench_phonons`** (پیش‌بینی یک عدد اسکالر: فرکانس
پیک آخر phonon DOS)، تا معلوم بشه در مقایسه با مدل‌های منتشرشده تقریباً کجا می‌ایستیم.

## ⚠️ تغییر مهم نسبت به نسخه‌ی قبلی این notebook
نصب پکیج رسمی `matbench` روی Kaggle (Python 3.12) با خطای `metadata-generation-failed`
شکست خورد (ناسازگاری build). به‌جایش:
- دیتا مستقیماً با `matminer.datasets.load_dataset("matbench_phonons")` لود می‌شود
  (همون دیتای زیرین که پکیج `matbench` هم داخلش از آن استفاده می‌کند).
- چون آبجکت رسمی `MatbenchTask` در دسترس نیست، **fold های دقیقاً رسمی و از پیش‌تعیین‌شده‌ی
  لیدربورد در دسترس ما نیستند**. به‌جایش یک 5-fold CV با `sklearn.model_selection.KFold`
  (seed ثابت) پیاده می‌کنیم که از نظر روش‌شناسی معادل است (همون معیار MAE، همون تعداد fold،
  بدون نشتی داده بین train/test)، اما دقیقاً همون اندیس‌های تقسیم رسمی لیدربورد نیست.
- **نتیجه:** عدد نهایی این notebook «قابل‌مقایسه از نظر روش‌شناسی» با جدول لیدربورد است،
  ولی رسماً قابل ثبت در خود سایت Matbench (submission) نیست — چون از پروتکل دقیق آن‌ها
  استفاده نشده. برای مقاله و مقایسه‌ی داخلی این کاملاً کافی است.

## تفاوت مهم این تسک با پروژه‌ی اصلی
| | پروژه‌ی اصلی (MAX Phase) | matbench_phonons |
|---|---|---|
| خروجی | کل ماتریس IFC → منحنی dispersion کامل (از طریق Phonopy) | **یک عدد اسکالر**: فرکانس پیک آخر phonon DOS |
| واحد | THz | **cm⁻¹** |
| تعداد نمونه | 358 | **1265** |
| نیاز به Phonopy در inference | بله | **خیر** |
| پروتکل ارزیابی | Train/Val/Test ساده (seed=42) | **5-fold CV (خودمان، معادل روش‌شناختی رسمی)** |

## معیار مقایسه (جمع‌آوری‌شده از لیدربورد و مقالات مرتبط)

| مدل | MAE (cm⁻¹) | یادداشت |
|---|---|---|
| CGCNN (baseline رسمی Matbench) | 57.76 ± 12.31 | پایین‌ترین سطح مقایسه |
| Matformer | 42.53 ± 11.89 | |
| MODNet | 34.28 ± 2.08 | |
| M3GNet | 34.16 ± 4.5 | |
| ALIGNN | 29.54 ± 2.12 | |
| coGN | 29.71 ± 2.0 | |
| coNGN | 28.89 ± 3.28 | |
| DenseGNN (بهترین نسخه) | ~23.5 | از قوی‌ترین‌های منتشرشده |
| TriForces (fine-tune روی eSEN) | ~19.5 | نزدیک SOTA فعلی |

⚠️ این اعداد از جست‌وجوی وب جمع‌آوری شده‌اند (نه از سایت لیدربورد که بلاک بود).


In [1]:
!pip install -q -U matminer
print("Installed: matminer")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 57.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 55.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 97.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 100.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 106.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/

In [3]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00a 0:00:01


In [15]:
import json as json_lib
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from tqdm.notebook import tqdm

from matminer.datasets import load_dataset
from sklearn.model_selection import KFold

device = torch.device('cpu' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


## 1. Load the matbench_phonons dataset (via matminer directly)

In [16]:
df = load_dataset("matbench_phonons")

structures = df["structure"].tolist()
targets = df["last phdos peak"].to_numpy().astype(np.float32)

print(f"Loaded {len(structures)} structures")
print(f"Target range: [{targets.min():.1f}, {targets.max():.1f}] cm^-1, mean={targets.mean():.1f}")

Loaded 1265 structures
Target range: [59.6, 3643.7] cm^-1, mean=581.4


## 2. Atomic feature table (same feature set as the main project: `5_geometry_radius_volume`)

⚠️ **نکته‌ی مهم:** این فیچرست در پروژه‌ی اصلی روی عناصر محدود فازهای MAX (M/A/X) تست شده
بود. دیتاست matbench_phonons عناصر بسیار متنوع‌تری دارد. زیر، پوشش این فیچرست روی عناصر
واقعی دیتاست matbench بررسی و گزارش می‌شود؛ اگر پوشش ضعیف بود، باید فیچرست جایگزین در نظر
گرفته شود.

In [17]:
FEATURE_CSV = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"Atomic features: {N_ATOM_FEATURES} -> {feature_cols}")
print(f"Elements covered by feature table: {len(df_features.index)}")

Atomic features: 7 -> ['atomic_weight', 'atomic_radius', 'atomic_radius_rahm', 'covalent_radius_cordero', 'vdw_radius', 'atomic_volume', 'lattice_constant']
Elements covered by feature table: 118


In [18]:
# Check element coverage against the actual matbench_phonons dataset
all_elements = set()
for struct in structures:
    for site in struct:
        all_elements.add(site.specie.symbol)

missing_elements = sorted(all_elements - set(df_features.index))
covered_elements = sorted(all_elements & set(df_features.index))
print(f"Total unique elements in matbench_phonons: {len(all_elements)}")
print(f"Covered by feature table: {len(covered_elements)}")
print(f"Missing (will fall back to zero-vector): {len(missing_elements)}")
if missing_elements:
    print(f"Missing elements: {missing_elements}")

Total unique elements in matbench_phonons: 64
Covered by feature table: 64
Missing (will fall back to zero-vector): 0


## 3. Graph construction (bond graph + atom graph) — same approach as the main project

In [19]:
CUTOFF = 8.0


def structure_to_bond_graph(positions, n_atoms_sample):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def structure_to_atom_graph(atom_elements, positions, n_atoms_sample):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def pymatgen_structure_to_graphs(structure):
    positions = structure.cart_coords.astype(np.float32)
    atom_elements = [site.specie.symbol for site in structure]
    n_atoms = len(atom_elements)
    bond_g = structure_to_bond_graph(positions, n_atoms)
    atom_g = structure_to_atom_graph(atom_elements, positions, n_atoms)
    return bond_g, atom_g


print("Graph construction functions ready (reused pattern from the main project's notebook 15)")

Graph construction functions ready (reused pattern from the main project's notebook 15)


## 4. Build all graphs once (upfront)

چون دیگر fold ها را خود `matbench` تعیین نمی‌کند، همه‌ی ۱۲۶۵ گراف یک‌بار ساخته می‌شوند و
بعد با اندیس‌های KFold به train/test هر fold تقسیم می‌شوند — این از ساخت مکرر گراف برای هر
fold جلوگیری می‌کند و سریع‌تر است.

In [20]:
bond_graphs, atom_graphs = [], []
for s in tqdm(structures, desc="Building graphs"):
    bg, ag = pymatgen_structure_to_graphs(s)
    bond_graphs.append(bg)
    atom_graphs.append(ag)

print(f"Built {len(bond_graphs)} bond graphs and {len(atom_graphs)} atom graphs")

Building graphs:   0%|          | 0/1265 [00:00<?, ?it/s]

Built 1265 bond graphs and 1265 atom graphs


## 5. Model: Dual Graph Regressor (بدون فیزیک، بدون Phonopy)

معماری انکودر دقیقاً همون الگوی دوگانه‌ی bond graph + atom graph پروژه‌ی اصلی است
(message passing + attention pooling + Set2Set/mean/max)، اما به‌جای سر پیش‌بینی
ماتریس IFC، یک MLP رگرسیون ساده خروجی یک عدد اسکالر (فرکانس پیک phonon DOS، بر حسب cm⁻¹)
می‌دهد.

In [21]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphRegressor(nn.Module):
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1,
                 hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1,
                 set2set_steps=1):
        super().__init__()
        hidden = hidden_dim

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_bond_layers)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_bond_layers)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_atom_layers)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_atom_layers)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=set2set_steps)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        combined_dim = 8 * hidden + hidden // 4

        # Regression head: single scalar output (frequency at last phonon PhDOS peak, cm^-1)
        self.regression_head = nn.Sequential(
            nn.Linear(combined_dim, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.LayerNorm(64), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)

        return self.regression_head(combined).squeeze(-1)

print("DualGraphRegressor ready")

DualGraphRegressor ready


## 6. Training / prediction utilities

In [22]:
def make_batches(indices, batch_size, shuffle, rng):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = rng.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch(model, targets, indices, optimizer, batch_size, shuffle, rng):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_batches = 0., 0
    for batch_idx in make_batches(indices, batch_size, shuffle, rng):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        y = torch.tensor([targets[i] for i in batch_idx], dtype=torch.float32, device=device)

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            pred = model(bond_batch, atom_batch)
            loss = F.l1_loss(pred, y)  # MAE loss, matches the matbench evaluation metric
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def predict(model, indices, batch_size=16):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i+batch_size]
            bond_list = [bond_graphs[j] for j in batch_idx]
            atom_list = [atom_graphs[j] for j in batch_idx]
            bond_batch = Batch.from_data_list(bond_list).to(device)
            atom_batch = Batch.from_data_list(atom_list).to(device)
            pred = model(bond_batch, atom_batch)
            preds.extend(pred.cpu().numpy().tolist())
    return preds


print("Training/prediction utilities ready")

Training/prediction utilities ready


## 7. اجرای 5-fold CV (خودمان، معادل روش‌شناختی پروتکل رسمی Matbench)

برای هر fold:
1. `KFold(n_splits=5, shuffle=True, random_state=42)` مجموعه‌ی ۱۲۶۵ تایی را به train/test
   تقسیم می‌کند
2. از سهم train همان fold، یک split داخلی 90/10 برای early stopping گرفته می‌شود
3. مدل از صفر آموزش داده می‌شود (نه fine-tune از fold قبلی)
4. روی test همان fold پیش‌بینی و MAE واقعی محاسبه می‌شود (چون هدف تست‌ها را داریم،
   برخلاف پروتکل رسمی که هدف test مخفی است، اینجا مستقیماً MAE را حساب می‌کنیم)

In [ ]:
EPOCHS = 300
PATIENCE = 40
BATCH_SIZE = 16
LR = 5e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.1
SEED = 42
N_FOLDS = 5

all_indices = np.arange(len(structures))
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_test_maes = []

for fold_num, (train_val_idx, test_idx) in enumerate(kfold.split(all_indices)):
    print(f"\n{'='*60}\nFold {fold_num}\n{'='*60}")
    print(f"Train+val: {len(train_val_idx)} | Test: {len(test_idx)}")

    rng = np.random.default_rng(SEED + fold_num)
    perm = rng.permutation(train_val_idx)
    n_val = max(1, int(VAL_FRACTION * len(perm)))
    val_idx = perm[:n_val].tolist()
    train_idx = perm[n_val:].tolist()

    torch.manual_seed(SEED + fold_num)
    model = DualGraphRegressor(
        n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1,
        hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1, set2set_steps=1
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        train_loss = run_epoch(model, targets, train_idx, optimizer, BATCH_SIZE, True, rng)
        val_loss = run_epoch(model, targets, val_idx, None, BATCH_SIZE, False, rng)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 20 == 0:
            print(f"  Epoch {epoch:4d} | Train MAE: {train_loss:.3f} | Val MAE: {val_loss:.3f} | Best: {best_val_loss:.3f}")

        if patience_counter >= PATIENCE:
            print(f"  Early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/matbench_phonons_fold{fold_num}.pt')

    test_predictions = np.array(predict(model, test_idx.tolist(), batch_size=BATCH_SIZE))
    test_true = targets[test_idx]
    test_mae = float(np.mean(np.abs(test_predictions - test_true)))
    fold_test_maes.append(test_mae)

    print(f"  Fold {fold_num} done. Best internal val MAE: {best_val_loss:.3f} | Test MAE: {test_mae:.3f} cm^-1")


Fold 0
Train+val: 1012 | Test: 253
  Epoch    0 | Train MAE: 590.963 | Val MAE: 511.031 | Best: 511.031
  Epoch   20 | Train MAE: 544.807 | Val MAE: 463.884 | Best: 463.884
  Epoch   40 | Train MAE: 466.192 | Val MAE: 386.213 | Best: 386.213
  Epoch   60 | Train MAE: 376.085 | Val MAE: 291.955 | Best: 291.955
  Epoch   80 | Train MAE: 309.555 | Val MAE: 229.308 | Best: 229.308
  Epoch  100 | Train MAE: 267.480 | Val MAE: 194.251 | Best: 194.251
  Epoch  120 | Train MAE: 232.235 | Val MAE: 156.119 | Best: 156.119
  Epoch  140 | Train MAE: 200.778 | Val MAE: 125.057 | Best: 125.057
  Epoch  160 | Train MAE: 181.474 | Val MAE: 113.991 | Best: 112.358
  Epoch  180 | Train MAE: 168.165 | Val MAE: 101.326 | Best: 101.326
  Epoch  200 | Train MAE: 147.653 | Val MAE: 87.780 | Best: 87.780
  Epoch  220 | Train MAE: 135.015 | Val MAE: 81.431 | Best: 77.148
  Epoch  240 | Train MAE: 117.132 | Val MAE: 64.886 | Best: 64.886
  Epoch  260 | Train MAE: 104.917 | Val MAE: 64.726 | Best: 59.010
  Epoc

## 8. نتیجه‌ی نهایی و مقایسه با لیدربورد

In [ ]:
fold_test_maes = np.array(fold_test_maes)
mae_mean = fold_test_maes.mean()
mae_std = fold_test_maes.std()

print("=== This project's 5-fold CV results (matbench_phonons) ===")
for i, m in enumerate(fold_test_maes):
    print(f"  Fold {i}: {m:.3f} cm^-1")
print(f"\n  MAE mean: {mae_mean:.3f} cm^-1")
print(f"  MAE std:  {mae_std:.3f}")

print("\n=== Comparison with published leaderboard-area results ===")
print(f"  CGCNN (official Matbench baseline) : 57.76 cm^-1")
print(f"  Matformer                          : 42.53 cm^-1")
print(f"  MODNet                             : 34.28 cm^-1")
print(f"  M3GNet                             : 34.16 cm^-1")
print(f"  ALIGNN                             : 29.54 cm^-1")
print(f"  coGN                               : 29.71 cm^-1")
print(f"  coNGN                              : 28.89 cm^-1")
print(f"  DenseGNN (best variant)            : ~23.5 cm^-1")
print(f"  TriForces (eSEN foundation model)  : ~19.5 cm^-1")
print(f"  --------------------------------------------------")
print(f"  This project's Dual Graph GNN      : {mae_mean:.2f} +/- {mae_std:.2f} cm^-1")

In [ ]:
results = {
    'fold_test_maes': fold_test_maes.tolist(),
    'mae_mean': float(mae_mean),
    'mae_std': float(mae_std),
    'n_folds': N_FOLDS,
    'protocol_note': 'Custom 5-fold CV via sklearn KFold (seed=42), NOT the official pre-defined matbench leaderboard folds.',
}
with open('/kaggle/working/matbench_phonons_results.json', 'w') as f:
    json_lib.dump(results, f, indent=2)
print("Saved: /kaggle/working/matbench_phonons_results.json")

## جمع‌بندی

این notebook مسیر مکملی نسبت به پروژه‌ی اصلی باز کرد: همون انکودر دوگانه‌ی گراف
(bond graph + atom graph) که برای پیش‌بینی IFC فازهای MAX ساخته شده بود، اینجا با یک سر
رگرسیون ساده روی تسک عمومی‌تر `matbench_phonons` (پیش‌بینی یک عدد اسکالر — فرکانس پیک
آخر phonon DOS — برای 1265 ماده‌ی عمومی) امتحان شد.

### نکات مهم برای تفسیر نتیجه
- چون پکیج رسمی `matbench` قابل نصب نبود، از یک 5-fold CV خودساخته (نه fold های دقیقاً
  رسمی لیدربورد) استفاده شد. عدد نهایی از نظر روش‌شناسی قابل‌مقایسه با جدول لیدربورد است
  اما رسماً «submission» به حساب نمی‌آید.
- این عدد MAE **قابل مقایسه‌ی مستقیم نیست** با MAE=0.909 (یا 0.8903) پروژه‌ی اصلی — دیتاست،
  واحد (cm⁻¹ در برابر THz)، و نوع خروجی (اسکالر در برابر منحنی کامل dispersion) متفاوتند.
- پوشش فیچرست اتمی روی عناصر واقعی matbench_phonons در بخش ۲ گزارش می‌شود.
- این اجرا از صفر و بدون هیچ tuning‌ای است؛ یعنی احتمالاً «حد پایین» عملکرد ممکن این
  معماری روی این تسک است، نه سقف آن.

### ثبت در Obsidian (بعد از اجرا روی Kaggle)
- عدد نهایی MAE (mean ± std روی ۵ فولد) و جایگاه آن نسبت به جدول بالا
- یادداشت این‌که پکیج `matbench` قابل‌نصب نبود و از رویکرد جایگزین matminer+KFold استفاده شد
- تصمیم درباره‌ی ادامه یا نه: آیا این مسیر ارزش سرمایه‌گذاری بیشتر (tuning، فیچرهای بهتر) دارد
